# 🌿 Darukaa Reference Benchmarking Pipeline

**Compare any biodiversity indicator against ecoregion-specific reference benchmarks, then compute the State of Nature Score per the State of Nature Module PRD v2.0.**

This notebook runs the full pipeline:
1. Clone the repo & install dependencies
2. Authenticate Google Earth Engine
3. Upload your KML site file
4. Configure and run Tier 1 (regional) + Tier 2 (contemporary) benchmarking
5. View the benchmark scorecard
6. Compute State of Nature concern levels and SoN Score
7. Download results

**Reference benchmarking methodology:**
- McElderry et al. (2024) DOI:10.32942/X2689N — SEED reference selection (adapted)
- McNellie et al. (2020) DOI:10.1111/gcb.15383 — Contemporary reference states
- Yen et al. (2019) DOI:10.1002/eap.1970 — Biodiversity benchmarks

**State of Nature scoring methodology:**
- Darukaa State of Nature Module PRD v2.0
- TNFD LEAP Annex 2 (Ecosystem Extent / Condition / Species Population / Extinction Risk)
- Normalized sum formula: `SoN_Score = (Σ dim_scores − 4) / 16 × 10`

## 1. Setup — Clone repo & install

In [15]:
# ── Setup: Clone or update repo ──────────────────────────
import os
os.chdir("/content")  # Always start from a safe location

REPO_DIR = "/content/reference-benchmarking"
CODE_DIR = f"{REPO_DIR}/darukaa_reference_v0.1.0"

if os.path.exists(REPO_DIR):
    os.chdir(CODE_DIR)
    !git pull origin main
    print("✓ Pulled latest changes")
else:
    !git clone https://@github.com/G-auravSingh/reference-benchmarking.git {REPO_DIR}
    os.chdir(CODE_DIR)
    print("✓ Cloned fresh")

!pip install -e . --force-reinstall -q 2>&1 | tail -1

import sys
cleared = [k for k in sys.modules if 'darukaa' in k]
for m in cleared:
    del sys.modules[m]

os.chdir("/content")
print(f"✓ Installed from {CODE_DIR}, cleared {len(cleared)} cached modules")

From https://github.com/G-auravSingh/reference-benchmarking
 * branch            main       -> FETCH_HEAD
Already up to date.
✓ Pulled latest changes
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 7.34.1 which is incompatible.
✓ Installed from /content/reference-benchmarking/darukaa_reference_v0.1.0, cleared 10 cached modules


## 2. Authenticate Google Earth Engine

Run the cell below — it will show a link. Click it, sign in with your Google account, copy the token back.

In [16]:
import ee

# Colab has a built-in GEE authenticator
ee.Authenticate()

# Initialize with your GEE project
# Replace with your actual project ID
GEE_PROJECT = "gaurav-singh-007"  # @param {type:"string"}

ee.Initialize(project=GEE_PROJECT)
print(f"✓ GEE initialised with project: {GEE_PROJECT}")

✓ GEE initialised with project: gaurav-singh-007


## 3. Upload your KML file

Run the cell below to upload your project site KML/KMZ/GeoJSON file.

In [6]:
from google.colab import files

print("Upload your KML/KMZ/GeoJSON file:")
uploaded = files.upload()

# Get the filename
SITE_FILE = list(uploaded.keys())[0]
print(f"\n✓ Uploaded: {SITE_FILE} ({len(uploaded[SITE_FILE])/1024:.1f} KB)")

Upload your KML/KMZ/GeoJSON file:


Saving reference_cluster_RC1.kml to reference_cluster_RC1.kml
Saving reference_cluster_RC2.kml to reference_cluster_RC2.kml
Saving reference_cluster_RC3.kml to reference_cluster_RC3.kml
Saving reference_cluster_RC4.kml to reference_cluster_RC4.kml
Saving reference_cluster_RC5.kml to reference_cluster_RC5.kml
Saving reference_cluster_RC6.kml to reference_cluster_RC6.kml
Saving reference_cluster_RC7.kml to reference_cluster_RC7.kml
Saving reference_cluster_RC8.kml to reference_cluster_RC8.kml
Saving reference_cluster_RC9.kml to reference_cluster_RC9.kml
Saving reference_cluster_RC10.kml to reference_cluster_RC10.kml
Saving reference_cluster_RC11.kml to reference_cluster_RC11.kml
Saving reference_cluster_RC12.kml to reference_cluster_RC12.kml
Saving reference_cluster_RC13.kml to reference_cluster_RC13.kml
Saving reference_cluster_RC14.kml to reference_cluster_RC14.kml
Saving reference_cluster_RC15.kml to reference_cluster_RC15.kml
Saving reference_cluster_RC16.kml to reference_cluster_RC1



```
# This is formatted as code
```

## 4. Configure the pipeline

Key parameters:
- `HMI_CEILING`: 0.10 (adapted from SEED's 0.05 for buffer-scale analysis — see README)
- `ELEVATION_BAND_M`: ±300m elevation stratification for Tier 2
- `NDVI_YEAR` / `LST_YEAR`: remote sensing year for temporal indicators
- `ENABLED_INDICATORS`: leave empty for all 25 registered indicators

In [19]:
# ── Configuration ─────────────────────────────────────────────────────────────

# ── Project type ──────────────────────────────────────────────────────────────
# "conservation" : single KML/KMZ site → runs exactly as before
# "agroforestry" : multiple cluster KMLs + manifest.json → aggregates internally
PROJECT_TYPE = "agroforestry"  # @param ["conservation", "agroforestry"]

# ── Trajectory tracking ───────────────────────────────────────────────────────
# IS_BASELINE_RUN = True  → Year 0. Saves baseline_scorecard.csv; no trajectory cols.
# IS_BASELINE_RUN = False → Year 1+. Loads baseline_scorecard.csv and computes Δ.
IS_BASELINE_RUN = True  # @param {type:"boolean"}

# For agroforestry only: path to manifest JSON with per-cluster area weights.
# Format: reference_clusters_manifest.json from the generalised site selection pipeline.
# Top-level keys: project_name, clusters (array), total_area_ha.
# Each cluster: {"cluster_label": "RC1", "kml_file": "/content/reference_cluster_RC1.kml", "area_ha": 20.04, ...}
# Leave as None for conservation projects.
MANIFEST_FILE = "/content/reference_clusters_manifest.json"  # e.g. "/content/manifest.json"

# Baseline file path (used for trajectory on Year 1+ runs)
BASELINE_FILE = "./output/baseline_scorecard.csv"

# ── Which indicators to run? Leave empty [] for all 25. ──────────────────────
ENABLED_INDICATORS = []  # e.g. ["ndvi", "bii", "eii"] to run a subset

# ── Tier 2 reference selection parameters ─────────────────────────────────────
HMI_CEILING = 0.30       # Adapted from SEED's 0.05 for buffer-scale analysis
ELEVATION_BAND_M = 300.0 # ±m elevation filter for Tier 2 stratification
MIN_REF_PIXELS = 5       # SEED minimum before fallback cascade

# ── Remote sensing year ───────────────────────────────────────────────────────
NDVI_YEAR = 2025  # @param {type:"integer"}
LST_YEAR  = 2025  # @param {type:"integer"}

# ── Output directory ──────────────────────────────────────────────────────────
OUTPUT_DIR = "./output"

# ─────────────────────────────────────────────────────────────────────────────
print(f"Project type:        {PROJECT_TYPE}")
print(f"Baseline run:        {IS_BASELINE_RUN}")
print(f"HMI ceiling:         {HMI_CEILING}")
print(f"Elevation band:      ±{ELEVATION_BAND_M}m")
print(f"Min ref pixels:      {MIN_REF_PIXELS}")
print(f"Remote sensing year: {NDVI_YEAR}")
print(f"Indicators:          {'all 25' if not ENABLED_INDICATORS else ENABLED_INDICATORS}")
if PROJECT_TYPE == "agroforestry":
    print(f"Manifest file:       {MANIFEST_FILE}")


Project type:        agroforestry
Baseline run:        True
HMI ceiling:         0.3
Elevation band:      ±300.0m
Min ref pixels:      5
Remote sensing year: 2025
Indicators:          all 25
Manifest file:       /content/reference_clusters_manifest.json


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 5. Run the pipeline

In [20]:
import logging, os, json as _json
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
import sys
sys.path.append("/content/reference-benchmarking/darukaa_reference_v0.1.0")
from darukaa_reference.config import Config
from darukaa_reference.indicators import create_default_registry
from darukaa_reference.pipeline import Pipeline

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Build shared config ────────────────────────────────────────────────────────
config = Config(
    gee_project=GEE_PROJECT,
    bii_gee_asset="projects/gaurav-singh-007/assets/bii-2020_v2-1-1",
    hmi_hard_ceiling=HMI_CEILING,
    elevation_band_m=ELEVATION_BAND_M,
    min_reference_pixels=MIN_REF_PIXELS,
    ndvi_year=NDVI_YEAR,
    lst_year=LST_YEAR,
    raster_paths={
        "iucn_mammals": "projects/darukaa-earth130226/assets/RedList_Mammals_Terrestrial",
        "iucn_birds":   "projects/darukaa-earth130226/assets/RedList_Bird_IUCN_Category",
        "kba_global":   "projects/darukaa-earth130226/assets/KBA_Global_POL_SEP25",
        "pv_binary":    "projects/darukaa-earth-product/assets/biodiversity_India_PV_Binary_2025_Full_Mosaic",
        "msa":          "projects/ee-jayankandir/assets/TerrestrialMSA_2015_World",
    },
    output_dir=OUTPUT_DIR,
    output_format="both",
    enabled_indicators=ENABLED_INDICATORS,
)

registry = create_default_registry()
print(f"\u2713 Registry loaded: {len(registry)} indicators")
t2 = registry.tier2_indicators()
print(f"  Tier 2 eligible: {len(t2)} indicators")
print(f"  Tier 2 excluded: {len(registry)-len(t2)} indicators (threats/protocol-A)")

# ── Run pipeline — conservation (single site) or agroforestry (cluster loop) ──
if PROJECT_TYPE == "conservation":
    # ── Standard single-site run ─────────────────────────────────────────────
    pipeline = Pipeline(config, registry)
    report   = pipeline.run(
        site_path=SITE_FILE,
        output_path=f"{OUTPUT_DIR}/benchmark_scorecard",
    )
    print(f"\n\u2713 Pipeline complete")

elif PROJECT_TYPE == "agroforestry":
    # ── Multi-cluster run with area-weighted aggregation ──────────────────────
    # Load manifest
    if MANIFEST_FILE is None:
        raise ValueError("MANIFEST_FILE must be set for agroforestry projects. "
                         "Set it in Cell 8 to the path of your manifest.json.")
    with open(MANIFEST_FILE) as _f:
        manifest_json = _json.load(_f)
    manifest = manifest_json["clusters"]  # actual key in reference_clusters_manifest.json
    print(f"Manifest loaded: {len(manifest)} clusters ({manifest_json.get('project_name','')})")

    # Validate: check all kml_path entries exist
    for entry in manifest:
        if not os.path.exists(entry["kml_file"]):
            raise FileNotFoundError(
                f"Cluster KML not found: {entry['kml_file']}\n"
                "Upload all cluster KMLs to /content/ before running."
            )

    # Run pipeline for each cluster, collect scorecards
    cluster_scorecards = []   # list of (scorecard_rows, area_ha)
    total_area = manifest_json.get("total_area_ha") or sum(e["area_ha"] for e in manifest)

    for i, entry in enumerate(manifest):
        cid   = entry["cluster_label"]
        kpath = entry["kml_file"]
        area  = entry["area_ha"]
        print(f"\n  [{i+1}/{len(manifest)}] Running cluster: {cid} ({area:.1f} ha) ...")

        pipeline = Pipeline(config, registry)
        r = pipeline.run(
            site_path=kpath,
            output_path=f"{OUTPUT_DIR}/benchmark_scorecard_{cid}",
        )
        cluster_scorecards.append((r["scorecard"], area))
        print(f"  \u2713 {cid} complete — {len(r['scorecard'])} indicators")

    # ── Area-weighted aggregation ─────────────────────────────────────────────
    # For each indicator: project_value = Σ(cluster_value × area) / Σ area
    # Intactness ratios aggregated the same way (weighted mean of ratios).
    # Tier 2 reference: pooled weighted mean across clusters where T2 succeeded.
    print(f"\nAggregating {len(manifest)} clusters (total {total_area:.1f} ha)...")

    import pandas as pd, numpy as np

    # Build per-cluster DataFrames
    frames = []
    for rows, area in cluster_scorecards:
        cdf = pd.DataFrame(rows)
        cdf["_area"] = area
        frames.append(cdf)
    all_df = pd.concat(frames, ignore_index=True)

    def weighted_mean(vals, weights):
        """Area-weighted mean, ignoring NaN."""
        vals    = np.array(vals, dtype=float)
        weights = np.array(weights, dtype=float)
        mask    = ~np.isnan(vals)
        if mask.sum() == 0: return np.nan
        return np.average(vals[mask], weights=weights[mask])

    aggregated_rows = []
    for ind_name, grp in all_df.groupby("indicator_name", sort=False):
        areas = grp["_area"].values
        sv    = weighted_mean(grp["site_value"].values,       areas)
        t1r   = weighted_mean(grp["tier1_reference"].values,  areas)
        t2r   = weighted_mean(grp["tier2_reference"].values,  areas)
        # Intactness: weighted mean of numeric intactness values (strip % for calc)
        def parse_int(x):
            if isinstance(x, float): return x
            if isinstance(x, str) and x.endswith("%"):
                try: return float(x.strip("%"))/100
                except: return np.nan
            try: return float(x)
            except: return np.nan
        t1i = weighted_mean([parse_int(v) for v in grp["tier1_intactness"].values], areas)
        t2i = weighted_mean([parse_int(v) for v in grp["tier2_intactness"].values], areas)
        n_t2 = grp["tier2_reference"].notna().sum()

        agg_row = dict(grp.iloc[0])  # carry metadata from first cluster row
        agg_row.update({
            "site_id":          f"project_aggregated",
            "site_value":       sv   if not np.isnan(sv) else None,
            "tier1_reference":  t1r  if not np.isnan(t1r) else None,
            "tier2_reference":  t2r  if not np.isnan(t2r) else None,
            "tier1_intactness": t1i  if not np.isnan(t1i) else None,
            "tier2_intactness": t2i  if not np.isnan(t2i) else None,
            "n_clusters_total": len(manifest),
            "n_clusters_with_data": int((~np.isnan(grp["site_value"].astype(float))).sum()),
            "n_clusters_tier2": int(n_t2),
            "total_area_ha":    total_area,
            "aggregation_note": f"Area-weighted mean across {len(manifest)} DBSCAN clusters",
            "_area":            total_area,
        })
        aggregated_rows.append(agg_row)

    # Build a synthetic report object that downstream cells expect
    report = {
        "scorecard": aggregated_rows,
        "project_type": "agroforestry",
        "n_clusters": len(manifest),
        "total_area_ha": total_area,
    }
    print(f"\u2713 Aggregation complete: {len(aggregated_rows)} indicators, "
          f"{len(manifest)} clusters, {total_area:.1f} ha total")

    # ── Tier 2 landscape note ─────────────────────────────────────────────────
    import pandas as pd
    agg_df = pd.DataFrame(aggregated_rows)
    n_t2_success = (agg_df["n_clusters_tier2"] > 0).sum()
    n_total = len(agg_df)
    if n_t2_success == 0:
        print("\n  Note: Tier 2 unavailable for all indicators — no minimally-modified")
        print("  reference pixels (HMI ≤ 0.10) found within any cluster buffer.")
        print("  This is expected behaviour for active agricultural landscapes.")
        print("  Tier 1 regional comparison applied throughout.")

else:
    raise ValueError(f"Unknown PROJECT_TYPE: {PROJECT_TYPE!r}. Use 'conservation' or 'agroforestry'.")


✓ Registry loaded: 25 indicators
  Tier 2 eligible: 13 indicators
  Tier 2 excluded: 12 indicators (threats/protocol-A)
Manifest loaded: 20 clusters (FCF_Orissa_Srishti)

  [1/20] Running cluster: RC1 (20.0 ha) ...


  ✓ RC1 complete — 25 indicators

  [2/20] Running cluster: RC2 (13.2 ha) ...


  ✓ RC2 complete — 25 indicators

  [3/20] Running cluster: RC3 (4.4 ha) ...


  ✓ RC3 complete — 25 indicators

  [4/20] Running cluster: RC4 (12.4 ha) ...


  ✓ RC4 complete — 25 indicators

  [5/20] Running cluster: RC5 (10.8 ha) ...


...

  ✓ RC17 complete — 25 indicators

  [18/20] Running cluster: RC18 (14.6 ha) ...


  ✓ RC18 complete — 25 indicators

  [19/20] Running cluster: RC19 (2.0 ha) ...


  ✓ RC19 complete — 25 indicators

  [20/20] Running cluster: RC20 (11.6 ha) ...


  ✓ RC20 complete — 25 indicators

Aggregating 20 clusters (total 724.6 ha)...


KeyError: 'indicator_name'

## 6. Benchmark Scorecard

Raw output: site values, Tier 1 (regional) and Tier 2 (contemporary least-disturbed) references, and intactness ratios for all indicators.

In [ ]:
import pandas as pd
from IPython.display import display

df = pd.DataFrame(report["scorecard"])

# For agroforestry: add aggregation metadata columns to display if present
_extra_cols = ["n_clusters_total", "n_clusters_with_data", "n_clusters_tier2",
               "total_area_ha", "aggregation_note"]
_agg_cols   = [c for c in _extra_cols if c in df.columns]
if _agg_cols:
    print(f"Project type: agroforestry — {report.get('n_clusters','')} clusters, "
          f"{report.get('total_area_ha',''):.1f} ha total")

display_cols = [
    "site_id", "display_name", "pillar", "site_value",
    "tier1_reference", "tier1_intactness",
    "tier2_reference", "tier2_intactness",
    "interpretation",
]
existing_cols = [c for c in display_cols if c in df.columns]

print(f"{'='*70}")
print(f"BENCHMARK SCORECARD: {len(df)} indicator-site pairs")
print(f"{'='*70}\n")

# Format for display
df_display = df[existing_cols].copy()
for col in ["site_value", "tier1_reference", "tier2_reference"]:
    if col in df_display:
        df_display[col] = df_display[col].apply(
            lambda x: f"{x:.4f}" if pd.notna(x) else "—"
        )
for col in ["tier1_intactness", "tier2_intactness"]:
    if col in df_display:
        df_display[col] = df_display[col].apply(
            lambda x: f"{x:.1%}" if pd.notna(x) else "—"
        )

pd.set_option('display.max_colwidth', 40)
pd.set_option('display.max_rows', 30)
display(df_display)

## 7. State of Nature Scoring

Converts intactness ratios into concern levels and computes the State of Nature Score per **Darukaa SoN Module PRD v2.0** and **TNFD LEAP Annex 2**.

**Three-tier scoring:**
- **Tier 1 (per-metric):** Intactness % → concern level (VL/L/M/H/VH) via threshold protocol
- **Tier 2 (per-dimension):** Mean concern numeric per TNFD dimension
- **Tier 3 (site-level):** `SoN_Score = (Σ dim_scores − n_dims) / (n_dims × 4) × 10` → 0–10 scale

**Protocol B v1.0 — Fixed, version-controlled thresholds (applied to intactness %):**
```
≥ 85% → VL (Very Low)    Basis: Newbold et al. 2016 planetary boundary (~90% BII)
70–84% → L  (Low)        Noticeably impacted, recoverable (SBTN science)
50–69% → M  (Moderate)   Substantially degraded — material loss (GLOBIO4 literature)
30–49% → H  (High)       Severely impaired
<  30% → VH (Very High)  Critically impaired
```
Review trigger: ≥10 Darukaa sites across ≥3 ecoregion types. These thresholds are fixed, not project-specific.

**Protocol A:** BII (NHM thresholds), FLII (Potapov thresholds) — fixed published literature values.

**Dimension scoring:** Computed from all indicators with a populated concern level — **no minimum metric constraints** at this stage. Missing metrics are shown as `n=X of Y` so confidence is transparent. The SoN PRD v2.0 data sufficiency rules will be enforced in the product UI layer once in-situ data is integrated.

**Threats** (gHM, light pollution, HDI, LST) are contextual only — not in SoN score per TNFD Annex 2.

In [ ]:
import numpy as np

# ── Threshold Tables ─────────────────────────────────────────────────────
# Numeric encoding: VL=1, L=2, M=3, H=4, VH=5

# Protocol A: Published literature thresholds (applied to raw site value)
# These indicators have defensible absolute thresholds — no intactness ratio needed.
PROTOCOL_A = {
    # ── Ecosystem Condition ──
    # BII: NHM / TNFD Annex 2 (higher=better, 0-1)
    "bii":  {"breaks": [0.20, 0.40, 0.60, 0.80], "scores": [5, 4, 3, 2, 1], "higher_is_better": True},
    # FLII: Potapov et al. 2020 (higher=better, 0-10)
    "flii": {"breaks": [2.0,  4.0,  6.0,  8.0],  "scores": [5, 4, 3, 2, 1], "higher_is_better": True},
    # MSA: GLOBIO4 (higher=better, 0-1) -- not yet populated
    "msa":  {"breaks": [0.20, 0.40, 0.60, 0.80], "scores": [5, 4, 3, 2, 1], "higher_is_better": True},

    # ── Species Population Size ──
    # Flagship Habitat Viability: HSI literature (higher=better, 0-1)
    # >0.8=VL (excellent), 0.6-0.8=L (good), 0.4-0.6=M (moderate), 0.2-0.4=H (poor), <0.2=VH
    "flagship_habitat": {"breaks": [0.20, 0.40, 0.60, 0.80], "scores": [5, 4, 3, 2, 1], "higher_is_better": True},

    # ── Species Extinction Risk ──
    # CERI: Butchart et al. 2007 (lower=better, 0-1)
    # 0-0.1=VL (LC dominated), 0.1-0.2=L, 0.2-0.35=M, 0.35-0.5=H, >0.5=VH
    "ceri": {"breaks": [0.10, 0.20, 0.35, 0.50], "scores": [1, 2, 3, 4, 5], "higher_is_better": False},

    # STAR_T: IUCN STAR methodology (lower=better, higher score = more abatement needed)
    # 0=VL, 1-3=L, 3-6=M, 6-9=H, >9=VH
    "star_t": {"breaks": [1.0, 3.0, 6.0, 9.0], "scores": [1, 2, 3, 4, 5], "higher_is_better": False},

    # KBA/IBA Overlap: TNFD/IBAT scheme (higher overlap = higher concern for disclosure)
    # Note: KBA site value = % overlap. 0%=VL, 1-25%=L, 25-75%=M, 75-99%=H, 100%=VH
    # Rationale: being inside a KBA signals high biodiversity importance (nature+ opportunity)
    # and high dependency/impact risk — TNFD treats this as elevated concern for disclosure
    "kba_overlap": {"breaks": [1.0, 25.0, 75.0, 99.9], "scores": [1, 2, 3, 4, 5], "higher_is_better": False},
}

# Protocol B v1.0 -- FIXED thresholds (applied to Tier 2 intactness %, or Tier 1 where T2 unavailable)
# Scientific basis:
#   >=85% (VL): Newbold et al. 2016 planetary boundary
#   70-84% (L): Noticeably impacted, recoverable (SBTN)
#   50-69% (M): Substantially degraded (GLOBIO4)
#   30-49% (H): Severely impaired
#    <30% (VH): Critically impaired
# Version: Protocol B v1.0 | Review trigger: >=10 sites across >=3 ecoregion types
PROTOCOL_B_BREAKS = [30, 50, 70, 85]
PROTOCOL_B_SCORES = [5,  4,  3,  2, 1]

# Protocol C: Tier 1 regional comparison — for count/connectivity indicators where
# absolute thresholds are ecoregion-dependent. Intactness = site / T1_regional_mean.
# These use the same Protocol B intactness % thresholds applied to T1 intactness.
# Indicators: natural_habitat*, natural_landcover*, cpland, forest_loss_rate*,
#             endemic_richness, threatened_richness
# (* also have T2, so C only applies when T2 fails)
PROTOCOL_C_INDICATORS = {"cpland", "endemic_richness", "threatened_richness"}

# TNFD Annex 2 dimension membership
DIM_MAP = {
    1: ["natural_habitat", "natural_landcover", "cpland", "forest_loss_rate", "kba_overlap"],
    2: ["ndvi", "habitat_health", "flii", "eii", "eii_structural",
        "eii_compositional", "eii_functional", "bii", "pdf", "aridity_index"],
    3: ["endemic_richness", "flagship_habitat"],
    4: ["threatened_richness", "ceri", "star_t"],
}
THREATS = ["ghm", "light_pollution", "hdi", "lst_day", "lst_night"]
DIM_NAMES = {
    1: "Ecosystem Extent",
    2: "Ecosystem Condition",
    3: "Species Population Size",
    4: "Species Extinction Risk",
}
PROTOCOL_A_INDICATORS = set(PROTOCOL_A.keys())


# ── Helpers ──────────────────────────────────────────────────────────────

def _is_valid(v):
    "True if v is a real finite number (not None, NaN, inf)."
    if v is None: return False
    try: return np.isfinite(float(v))
    except: return False

def apply_protocol_a(indicator_name, site_value):
    "Apply published absolute threshold. Returns concern numeric 1-5 or None."
    if not _is_valid(site_value): return None
    spec = PROTOCOL_A[indicator_name]
    val = float(site_value)
    # For lower-is-better: scores list already inverted (lower val = lower score = VL)
    for i, b in enumerate(spec["breaks"]):
        if val < b: return spec["scores"][i]
    return spec["scores"][-1]

def apply_protocol_b(intactness_ratio):
    "Protocol B v1.0 / C: intactness ratio 0-1 -> concern numeric."
    if not _is_valid(intactness_ratio): return None
    pct = float(intactness_ratio) * 100
    for i, b in enumerate(PROTOCOL_B_BREAKS):
        if pct < b: return PROTOCOL_B_SCORES[i]
    return PROTOCOL_B_SCORES[-1]

def concern_label(numeric):
    if numeric is None: return "—"
    return {1: "VL", 2: "L", 3: "M", 4: "H", 5: "VH"}.get(round(numeric), "—")

def dim_label(score):
    if score is None: return "—"
    s = round(score * 2) / 2
    if s <= 1.5: return "Very Low"
    if s <= 2.5: return "Low"
    if s <= 3.5: return "Moderate"
    if s <= 4.5: return "High"
    return "Very High"


# ── Build results lookup ──────────────────────────────────────────────────
results_lookup = {}
for _, row in df.iterrows():
    name = row.get("indicator_name") or row.get("name")
    if name is None:
        for spec in registry.all():
            if spec.display_name == row.get("display_name"):
                name = spec.name; break
    if name:
        results_lookup[name] = row.to_dict()


# ── Per-metric concern assignment ────────────────────────────────────────
# Priority order per indicator:
#   1. Protocol A (published absolute threshold) — BII, FLII, CERI, KBA, Flagship, STAR_T
#   2. Protocol B v1.0 on Tier 2 intactness    — all other indicators with T2
#   3. Protocol C (= B thresholds on Tier 1)   — CPLAND, endemic, threatened when T2 absent
#   4. Protocol B on Tier 1 fallback            — any remaining indicator with T1
#   5. No reference                             — concern = None, reported as "no ref"

metric_concerns = {}

for name, row in results_lookup.items():
    if name in THREATS:
        continue

    site_val = row.get("site_value")
    t2       = row.get("tier2_intactness")
    t1       = row.get("tier1_intactness")

    if not _is_valid(site_val): site_val = None
    if not _is_valid(t2):       t2 = None
    if not _is_valid(t1):       t1 = None

    spec_obj = registry.get(name) if name in registry else None
    hib = spec_obj.higher_is_better if spec_obj else True

    if name in PROTOCOL_A_INDICATORS and site_val is not None:
        # Protocol A: published absolute threshold
        cn = apply_protocol_a(name, site_val)
        protocol = "A (published absolute threshold)"
        intactness_used = None
        t_ref = None

    elif t2 is not None:
        # Protocol B v1.0: Tier 2 intactness (preferred)
        cn = apply_protocol_b(t2)
        protocol = "B v1.0 (Tier 2 intactness)"
        intactness_used = t2
        t_ref = row.get("tier2_reference")

    elif t1 is not None:
        # Protocol B/C on Tier 1: designed reference for count/connectivity indicators,
        # or fallback for raster indicators where T2 failed
        cn = apply_protocol_b(t1)
        if name in PROTOCOL_C_INDICATORS:
            protocol = "C (Tier 1 regional — designed ref for this indicator)"
        else:
            protocol = "B v1.0 (Tier 1 fallback — T2 unavailable)"
        intactness_used = t1
        t_ref = row.get("tier1_reference")

    else:
        # Genuinely no reference of any kind
        cn = None
        protocol = "— (no reference computable)"
        intactness_used = None
        t_ref = None

    metric_concerns[name] = {
        "concern_numeric":  cn,
        "concern_label":    concern_label(cn),
        "protocol":         protocol,
        "intactness_used":  intactness_used,
        "site_value":       site_val,
        "display_name":     row.get("display_name", name),
    }

n_scored  = sum(1 for v in metric_concerns.values() if v["concern_numeric"] is not None)
n_nodata  = sum(1 for v in metric_concerns.values() if v["concern_numeric"] is None)
n_prot_a  = sum(1 for v in metric_concerns.values() if "A" in v["protocol"])
n_prot_b  = sum(1 for v in metric_concerns.values() if "B v1.0" in v["protocol"] and "C" not in v["protocol"])
n_prot_c  = sum(1 for v in metric_concerns.values() if "C" in v["protocol"])
print(f"\u2713 Per-metric concern levels assigned")
print(f"  Protocol A (published):    {n_prot_a}")
print(f"  Protocol B v1.0 (T2):      {n_prot_b}")
print(f"  Protocol C / B-T1:         {n_prot_c}")
print(f"  No reference:              {n_nodata}")
print(f"  Total scored:              {n_scored} / {len(metric_concerns)}")


✓ Per-metric concern levels assigned
  Protocol A (published):    6
  Protocol B v1.0 (T2):      11
  Protocol C / B-T1:         3
  No reference:              0
  Total scored:              20 / 20


In [ ]:
# ── Per-metric concern table ─────────────────────────────────────────────
print(f"\n{'='*92}")
print(f"PER-METRIC CONCERN LEVELS  (TNFD Annex 2 | Protocol B v1.0)")
print(f"{'='*92}")
print(f"  {'Dim':<5} {'Indicator':<42} {'Site Value':<12} {'Intactness':<12} {'Concern':<8} Protocol")
print("  " + "-"*90)

for dim_num, indicators in DIM_MAP.items():
    print(f"\n  ── Dim {dim_num}: {DIM_NAMES[dim_num]} ──")
    for name in indicators:
        mc  = metric_concerns.get(name)
        row = results_lookup.get(name, {})
        dname = (mc["display_name"] if mc else row.get("display_name", name))[:41]

        if mc is None:
            print(f"  {dim_num:<5} {dname:<42} {'—':<12} {'—':<12} {'—':<8} not run")
            continue

        sv = mc["site_value"]
        sv_str = f"{sv:.4f}" if sv is not None else "—"

        iu = mc["intactness_used"]
        if iu is not None:
            intact_str = f"{iu:.1%}"
        elif mc["protocol"] == "A (published)":
            intact_str = "n/a (raw)"
        else:
            intact_str = "no ref"  # site value exists but no benchmark

        print(f"  {dim_num:<5} {dname:<42} {sv_str:<12} {intact_str:<12} {mc['concern_label']:<8} {mc['protocol']}")

print(f"\n  ── Threats (contextual — not in SoN score) ──")
for name in THREATS:
    row   = results_lookup.get(name, {})
    dname = row.get("display_name", name)[:41]
    sv    = row.get("site_value")
    t1    = row.get("tier1_intactness")
    sv_str = f"{float(sv):.4f}" if _is_valid(sv) else "—"
    t1_str = f"{float(t1):.1%}"  if _is_valid(t1) else "—"
    print(f"  {'T':<5} {dname:<42} {sv_str:<12} {t1_str:<12} {'—':<8} Tier 1 contextual")



PER-METRIC CONCERN LEVELS  (TNFD Annex 2 | Protocol B v1.0)
  Dim   Indicator                                  Site Value   Intactness   Concern  Protocol
  ------------------------------------------------------------------------------------------

  ── Dim 1: Ecosystem Extent ──
  1     Natural Habitat Extent                     95.9952      96.0%        VL       B v1.0 (Tier 2 intactness)
  1     Natural Land Cover Proportion              87.4370      87.4%        VL       B v1.0 (Tier 2 intactness)
  1     Landscape Connectivity (CPLAND)            88.1717      100.0%       VL       C (Tier 1 regional — designed ref for this indicator)
  1     Habitat Loss Rate                          0.2781       100.0%       VL       B v1.0 (Tier 2 intactness)
  1     KBA/IBA Overlap                            100.0000     no ref       VH       A (published absolute threshold)

  ── Dim 2: Ecosystem Condition ──
  2     Vegetation Structure (NDVI)                0.6298       72.9%        L      

In [ ]:
# ── Dimension scores & SoN Score ─────────────────────────────────────────
# Dimension score = mean concern numeric of all indicators with populated data.
# No minimum metric constraints — partial coverage shown via n_pop/n_total.
# Data sufficiency rules (SoN PRD v2.0) enforced in product UI, not here.

dim_scores   = {}
dim_populated = {}

for dim_num, indicators in DIM_MAP.items():
    values, populated_names = [], []
    for name in indicators:
        mc = metric_concerns.get(name)
        if mc and mc["concern_numeric"] is not None:
            values.append(mc["concern_numeric"])
            populated_names.append(name)
    dim_populated[dim_num] = populated_names
    dim_scores[dim_num] = sum(values) / len(values) if values else None


# ── SoN Score: normalized sum (SoN PRD v2.0) ─────────────────────────────
# Full formula (4 dims): (Sigma - 4) / 16 * 10
# Partial (<4 dims):     (Sigma - n) / (n*4) * 10,  flagged as partial

valid_dims = {d: s for d, s in dim_scores.items() if s is not None}
n_valid    = len(valid_dims)

if n_valid >= 1:
    total     = sum(valid_dims.values())
    son_score = (total - n_valid) / (n_valid * 4) * 10
    partial   = n_valid < 4
else:
    son_score = None
    partial   = True

def son_concern_label(score):
    if score is None: return "Insufficient data"
    if score < 4: return "Very Low"
    if score < 5: return "Low"
    if score < 7: return "Moderate"
    if score < 8: return "High"
    return "Very High"


# ── Dimension summary ─────────────────────────────────────────────────────
print(f"\n{'='*72}")
print(f"DIMENSION SCORES  (TNFD Annex 2 | Protocol B v1.0)")
print(f"{'='*72}")
print(f"  {'Dim':<5} {'Dimension':<30} {'Score':<7} {'Concern':<15} Coverage")
print("  " + "-"*70)

for dim_num in [1, 2, 3, 4]:
    dname  = DIM_NAMES[dim_num]
    score  = dim_scores.get(dim_num)
    n_pop  = len(dim_populated.get(dim_num, []))
    n_tot  = len(DIM_MAP[dim_num])
    s_str  = f"{score:.2f}" if score is not None else "—"
    label  = dim_label(score) if score is not None else "No data"
    flag   = "  ← in-situ metrics missing" if n_pop < n_tot else ""
    print(f"  {dim_num:<5} {dname:<30} {s_str:<7} {label:<15} {n_pop}/{n_tot}{flag}")


# ── SoN Score ────────────────────────────────────────────────────────────
print(f"\n{'='*72}")
print(f"STATE OF NATURE SCORE  (SoN PRD v2.0)")
print(f"{'='*72}")

if son_score is not None:
    print(f"  SoN Score:   {son_score:.1f} / 10")
    print(f"  Concern:     {son_concern_label(son_score)}")
    print(f"  Dims used:   {n_valid} of 4")
    if partial:
        missing = [DIM_NAMES[d] for d in [1,2,3,4] if dim_scores.get(d) is None]
        print(f"  Note: Partial score — no computable data for: {', '.join(missing)}")
        print(f"        Add in-situ species richness to complete Dim 3.")
    print()
    print(f"  Formula: ({total:.2f} - {n_valid}) / ({n_valid} x 4) x 10 = {son_score:.1f}")
    print()
    print(f"  Protocol B v1.0: >=85%=VL | 70-84%=L | 50-69%=M | 30-49%=H | <30%=VH")
    print(f"  Basis: Newbold et al. (2016) + GLOBIO4/SBTN literature")
    print(f"  Review trigger: >=10 Darukaa sites across >=3 ecoregion types")
else:
    print(f"  No dimension has computable data — check pipeline output.")


# ── Trajectory layer ──────────────────────────────────────────────────────────
# Year 0 (IS_BASELINE_RUN=True):  writes baseline_scorecard.csv; trajectory cols blank.
# Year 1+ (IS_BASELINE_RUN=False): loads baseline, computes delta + trajectory label.
#
# trajectory_label thresholds (SBTN AR3T Step 4 guidance):
#   Raster indices (NDVI, EII, BII, FLII, CPLAND, PDF, AI, HHI): ±2% intactness
#   Count indicators (endemic_richness, threatened_richness, ceri, star_t): ±0.05 absolute
#   Applicable to site_value change; intactness delta used for Protocol B indicators.

import pandas as pd, numpy as np, os

COUNT_INDICATORS = {"endemic_richness", "threatened_richness"}
ABS_DELTA_INDICATORS = {"ceri", "star_t", "flagship_habitat", "bii", "flii"}  # use site_value delta

def trajectory_label(delta, indicator_name):
    """
    Returns 'Improving', 'Stable', or 'Declining'.
    Delta is the change in Tier 2 intactness (for Protocol B) or site_value (for Protocol A/count).
    """
    if delta is None or np.isnan(delta): return None
    if indicator_name in COUNT_INDICATORS:
        # Count metrics: even 1 species matters
        threshold = 0.05
    elif indicator_name in ABS_DELTA_INDICATORS:
        threshold = 0.02  # 2% absolute change in site_value
    else:
        threshold = 0.02  # 2% intactness change
    if delta > threshold:  return "Improving"
    if delta < -threshold: return "Declining"
    return "Stable"

# Build scorecard with trajectory columns
df_traj = df[["indicator_name","site_value","tier1_intactness","tier2_intactness"]].copy()
df_traj["delta_site_value"]   = None
df_traj["delta_intactness"]   = None
df_traj["trajectory_label"]   = None
df_traj["trajectory_year"]    = None

if IS_BASELINE_RUN:
    # Write baseline — simple CSV of indicator_name → site_value, intactness values
    os.makedirs(os.path.dirname(BASELINE_FILE) if os.path.dirname(BASELINE_FILE) else ".", exist_ok=True)
    baseline_out = df[["indicator_name","site_value","tier1_intactness","tier2_intactness"]].copy()
    baseline_out.to_csv(BASELINE_FILE, index=False)
    print(f"\n{'='*70}")
    print(f"TRAJECTORY: BASELINE RUN (Year 0)")
    print(f"{'='*70}")
    print(f"  ✓ Baseline saved → {BASELINE_FILE}")
    print(f"  Set IS_BASELINE_RUN = False on next annual run to compute trajectory.")

else:
    if not os.path.exists(BASELINE_FILE):
        print(f"\n  WARNING: BASELINE_FILE not found at {BASELINE_FILE}")
        print("  Trajectory columns will be blank. Run with IS_BASELINE_RUN=True first.")
    else:
        baseline_df = pd.read_csv(BASELINE_FILE)
        baseline_df = baseline_df.rename(columns={
            "site_value":      "baseline_site_value",
            "tier1_intactness":"baseline_t1_intactness",
            "tier2_intactness":"baseline_t2_intactness",
        })
        df_traj = df_traj.merge(baseline_df, on="indicator_name", how="left")

        def _parse(v):
            if isinstance(v, float): return v
            if isinstance(v, str):
                v = v.strip().replace("%","")
                try: return float(v)/100 if float(v)>1 else float(v)
                except: return np.nan
            try: return float(v)
            except: return np.nan

        for idx, row in df_traj.iterrows():
            ind = row["indicator_name"]
            sv_now  = _parse(row["site_value"])
            sv_base = _parse(row.get("baseline_site_value"))
            t2_now  = _parse(row["tier2_intactness"])
            t2_base = _parse(row.get("baseline_t2_intactness"))
            t1_now  = _parse(row["tier1_intactness"])
            t1_base = _parse(row.get("baseline_t1_intactness"))

            # Delta site value (always computed)
            dsv = sv_now - sv_base if (sv_now is not None and sv_base is not None
                                       and not np.isnan(sv_now) and not np.isnan(sv_base)) else None
            # Delta intactness: prefer Tier 2, fall back to Tier 1
            if not (np.isnan(t2_now) if t2_now is not None else True) and                not (np.isnan(t2_base) if t2_base is not None else True):
                di = t2_now - t2_base
                int_tier = "T2"
            elif not (np.isnan(t1_now) if t1_now is not None else True) and                  not (np.isnan(t1_base) if t1_base is not None else True):
                di = t1_now - t1_base
                int_tier = "T1"
            else:
                di = None; int_tier = None

            # Choose delta for trajectory label
            if ind in ABS_DELTA_INDICATORS or ind in COUNT_INDICATORS:
                traj_delta = dsv
            else:
                traj_delta = di

            df_traj.at[idx, "delta_site_value"] = dsv
            df_traj.at[idx, "delta_intactness"]  = di
            df_traj.at[idx, "trajectory_label"]  = trajectory_label(traj_delta, ind)
            df_traj.at[idx, "trajectory_year"]   = "Year 1+"

        print(f"\n{'='*70}")
        print(f"TRAJECTORY: YEAR 1+ (Δ vs baseline)")
        print(f"{'='*70}")
        improving = (df_traj["trajectory_label"]=="Improving").sum()
        stable    = (df_traj["trajectory_label"]=="Stable").sum()
        declining = (df_traj["trajectory_label"]=="Declining").sum()
        print(f"  Improving: {improving} indicators")
        print(f"  Stable:    {stable} indicators")
        print(f"  Declining: {declining} indicators\n")

        traj_display = df_traj[["indicator_name","site_value","delta_site_value",
                                 "delta_intactness","trajectory_label"]].copy()
        traj_display = traj_display[traj_display["trajectory_label"].notna()]
        print(traj_display.to_string(index=False))



DIMENSION SCORES  (TNFD Annex 2 | Protocol B v1.0)
  Dim   Dimension                      Score   Concern         Coverage
  ----------------------------------------------------------------------
  1     Ecosystem Extent               1.80    Low             5/5
  2     Ecosystem Condition            2.00    Low             10/10
  3     Species Population Size        3.00    Moderate        2/2
  4     Species Extinction Risk        1.00    Very Low        3/3

STATE OF NATURE SCORE  (SoN PRD v2.0)
  SoN Score:   2.4 / 10
  Concern:     Very Low
  Dims used:   4 of 4

  Formula: (7.80 - 4) / (4 x 4) x 10 = 2.4

  Protocol B v1.0: >=85%=VL | 70-84%=L | 50-69%=M | 30-49%=H | <30%=VH
  Basis: Newbold et al. (2016) + GLOBIO4/SBTN literature
  Review trigger: >=10 Darukaa sites across >=3 ecoregion types


In [ ]:
# ── Dimension concern profile (count of indicators per concern level) ─────
print(f"\n{'='*70}")
print(f"DIMENSION CONCERN PROFILES")
print(f"(count of indicators at each concern level within each dimension)")
print(f"{'='*70}")
print(f"  This is the primary output for TNFD LEAP Step 3 disclosure.")
print(f"  Concerns: VL=Very Low · L=Low · M=Moderate · H=High · VH=Very High\n")

labels_order = ["VL", "L", "M", "H", "VH", "—"]

for dim_num, indicators in DIM_MAP.items():
    counts = {l: [] for l in labels_order}
    for name in indicators:
        mc = metric_concerns.get(name)
        cl = mc["concern_label"] if mc else "—"
        if cl not in counts:
            cl = "—"
        counts[cl].append(mc["display_name"] if mc else name)

    print(f"  Dim {dim_num} — {DIM_NAMES[dim_num]}")
    for level in labels_order:
        items = counts[level]
        if not items: continue
        indicator_names = ", ".join(items)
        print(f"    {level:>2}: {len(items)} — {indicator_names}")
    print()

# ── Threat / Pressure Layer — Categorical Labels ─────────────────────────
# Threats are excluded from SoN score but each has its own absolute concern level.
# These compare the raw site value against published benchmarks, not against
# a spatial reference — because there is no "least-disturbed" reference for pressure.

THREAT_THRESHOLDS = {
    # gHM: Kennedy et al. (2019) — 0-1 scale, lower=better
    "ghm": {
        "breaks": [0.10, 0.25, 0.40, 0.60],
        "labels": ["Very Low pressure", "Low pressure", "Moderate pressure",
                   "High pressure", "Very High pressure"],
        "unit": "(0-1)",
        "source": "Kennedy et al. 2019",
    },
    # Light Pollution (VIIRS nW/cm²/sr): Falchi et al. (2016) — lower=better
    "light_pollution": {
        "breaks": [0.05, 0.5, 5.0, 50.0],
        "labels": ["Natural dark sky", "Low light pollution", "Moderate",
                   "High — suburban influence", "Very High — urban core"],
        "unit": "nW/cm²/sr",
        "source": "Falchi et al. 2016",
    },
    # HDI: ESA WorldCover urban distance proxy — lower=better
    "hdi": {
        "breaks": [0.1, 0.3, 0.6, 0.8],
        "labels": ["Very Low disturbance", "Low", "Moderate",
                   "High", "Very High — urban proximity"],
        "unit": "(0-1)",
        "source": "ESA WorldCover v200",
    },
    # LST Day/Night: Report as deviation from regional Tier 1 mean
    # Not absolute threshold — region-specific
    "lst_day":   {"deviation_based": True, "unit": "°C"},
    "lst_night": {"deviation_based": True, "unit": "°C"},
}

def threat_label(name, site_val, t1_ref=None):
    if site_val is None or not _is_valid(site_val): return "—"
    spec = THREAT_THRESHOLDS.get(name)
    if spec is None: return "—"
    if spec.get("deviation_based"):
        if t1_ref is None or not _is_valid(t1_ref): return f"{float(site_val):.1f}°C (no regional ref)"
        dev = float(site_val) - float(t1_ref)
        sign = "+" if dev >= 0 else ""
        severity = "within normal" if abs(dev) < 1.0 else ("mildly elevated" if abs(dev) < 2.0 else "notably elevated")
        return f"{float(site_val):.1f}°C ({sign}{dev:.1f}°C vs regional mean — {severity})"
    val = float(site_val)
    for i, b in enumerate(spec["breaks"]):
        if val < b: return spec["labels"][i]
    return spec["labels"][-1]

print(f"\n{'='*70}")
print(f"THREATS & PRESSURES — Categorical Assessment")
print(f"(Contextual layer — not included in SoN score)")
print(f"{'='*70}")
print(f"  {'Indicator':<35} {'Value':<12} {'Assessment'}")
print("  " + "-"*68)

for name in THREATS:
    row = results_lookup.get(name, {})
    dname = row.get("display_name", name)[:34]
    sv = row.get("site_value")
    t1 = row.get("tier1_intactness")
    t1_ref = row.get("tier1_reference")
    sv_str = f"{float(sv):.4f}" if _is_valid(sv) else "—"
    label = threat_label(name, sv, t1_ref)
    print(f"  {dname:<35} {sv_str:<12} {label}")

print(f"\n  Note: gHM, Light Pollution, and HDI use absolute published thresholds.")
print(f"  LST Day/Night reported as deviation from Tier 1 regional mean (±°C).")



DIMENSION CONCERN PROFILES
(count of indicators at each concern level within each dimension)
  This is the primary output for TNFD LEAP Step 3 disclosure.
  Concerns: VL=Very Low · L=Low · M=Moderate · H=High · VH=Very High

  Dim 1 — Ecosystem Extent
    VL: 4 — Natural Habitat Extent, Natural Land Cover Proportion, Landscape Connectivity (CPLAND), Habitat Loss Rate
    VH: 1 — KBA/IBA Overlap

  Dim 2 — Ecosystem Condition
    VL: 5 — Ecosystem Integrity Index, EII: Structural Integrity, EII: Compositional Integrity, EII: Functional Integrity, Biodiversity Intactness Index
     L: 3 — Vegetation Structure (NDVI), Forest Landscape Integrity Index, Aridity Index
     H: 1 — Potentially Disappeared Fraction
    VH: 1 — Habitat Health Index (HHI)

  Dim 3 — Species Population Size
    VL: 1 — Flagship Habitat Viability
    VH: 1 — Endemic Species Richness

  Dim 4 — Species Extinction Risk
    VL: 3 — Threatened Species Richness, Composite Extinction-Risk Index, STAR_T (Threat Abatement

In [ ]:
# ── Species Lists for Reporting ─────────────────────────────────────────
# Re-runs extraction functions directly to retrieve species lists from IUCN range maps.
# Uses fastkml/geopandas to parse the uploaded KML and get site geometry.

import ee
from darukaa_reference.indicators import (
    extract_threatened_richness,
    extract_ceri,
    extract_endemic_richness,
)

# ── Get site geometry from the uploaded KML file ──────────────────────────
# Parse KML directly — no dependency on pipeline internals
def _kml_to_ee_geometries(kml_path):
    """Parse KML/KMZ and return dict of {site_id: ee.Geometry}."""
    import os
    from shapely.geometry import mapping
    from shapely.ops import transform as shapely_transform

    geoms = {}

    # Handle KMZ (zip containing KML)
    if kml_path.lower().endswith('.kmz'):
        import zipfile, io
        with zipfile.ZipFile(kml_path, 'r') as z:
            kml_names = [n for n in z.namelist() if n.lower().endswith('.kml')]
            kml_bytes = z.read(kml_names[0])
        kml_str = kml_bytes.decode('utf-8', errors='replace')
    else:
        with open(kml_path, 'rb') as f:
            kml_str = f.read().decode('utf-8', errors='replace')

    # Try fastkml first, then lxml fallback
    try:
        from fastkml import kml as fkml
        k = fkml.KML()
        k.from_string(kml_str.encode())
        features = list(list(k.features())[0].features())
        for i, feat in enumerate(features):
            name = getattr(feat, 'name', None) or f"site_{i:04d}"
            geom = feat.geometry
            if geom is None: continue
            # Strip Z coordinates
            if geom.has_z:
                geom = shapely_transform(lambda x,y,z=None:(x,y), geom)
            geoms[f"{name}_{i:04d}"] = ee.Geometry(mapping(geom))
    except Exception:
        # lxml fallback
        from lxml import etree
        from shapely.geometry import Polygon, MultiPolygon
        root = etree.fromstring(kml_str.encode())
        ns = {'kml': 'http://www.opengis.net/kml/2.2'}
        placemarks = root.findall('.//kml:Placemark', ns)
        for i, pm in enumerate(placemarks):
            name_el = pm.find('kml:name', ns)
            name = name_el.text if name_el is not None else f"site_{i:04d}"
            coords_el = pm.find('.//kml:coordinates', ns)
            if coords_el is None: continue
            coords_text = coords_el.text.strip()
            coords = []
            for c in coords_text.split():
                parts = c.split(',')
                if len(parts) >= 2:
                    coords.append((float(parts[0]), float(parts[1])))
            if len(coords) >= 3:
                geom = Polygon(coords)
                geoms[f"{name}_{i:04d}"] = ee.Geometry(mapping(geom))

    return geoms

print("Parsing site geometry from KML...")
try:
    site_geoms = _kml_to_ee_geometries(SITE_FILE)
    print(f"✓ Found {len(site_geoms)} site(s): {list(site_geoms.keys())}")
except Exception as e:
    print(f"✗ KML parse failed: {e}")
    site_geoms = {}


SPECIES_INDICATORS = {
    "threatened_richness": ("Threatened Species (CR/EN/VU) — mammals + birds", extract_threatened_richness),
    "ceri":                ("CERI Species — all IUCN categories (EX/EW/CR/EN/VU/NT/LC)", extract_ceri),
    "endemic_richness":    ("Endemic Species (range < 100,000 km²) — mammals + birds", extract_endemic_richness),
}

print(f"\n{'='*70}")
print(f"SPECIES LISTS (for reporting transparency)")
print(f"{'='*70}")
print(f"  Source: IUCN Red List range maps (mammals + birds)")
print(f"  CR=Critically Endangered, EN=Endangered, VU=Vulnerable, NT=Near Threatened")

all_species_data = {}

for site_id, geom in site_geoms.items():
    print(f"\n  Site: {site_id}")
    site_species = {}

    for ind_name, (label, extract_fn) in SPECIES_INDICATORS.items():
        print(f"\n  {label}")
        try:
            result = extract_fn(geom, config)
            meta = result.get("metadata", {})
            species_list = meta.get("species_list", [])

            if not species_list:
                print(f"    No species returned (check GEE asset access or re-install __init__.py)")
                continue

            mammals = sorted([s for s in species_list if s.get("group") == "mammal"],
                             key=lambda x: x.get("category",""))
            birds   = sorted([s for s in species_list if s.get("group") == "bird"],
                             key=lambda x: x.get("category",""))

            if mammals:
                print(f"\n    Mammals ({len(mammals)}):")
                for s in mammals:
                    print(f"      [{s.get('category','?'):2s}] {s['name']}")
            if birds:
                print(f"\n    Birds ({len(birds)}):")
                for s in birds:
                    print(f"      [{s.get('category','?'):2s}] {s['name']}")

            site_species[ind_name] = species_list

        except Exception as e:
            print(f"    Error running {ind_name}: {e}")

    all_species_data[site_id] = site_species

# Save for download
import json as _json
species_path = f"{OUTPUT_DIR}/species_lists.json"
with open(species_path, "w") as _f:
    _json.dump(all_species_data, _f, indent=2)
print(f"\n\u2713 Species lists saved → {species_path}")


Parsing site geometry from KML...
✓ Found 1 site(s): ['MER_CR_Demarcation2025_final_0000']

SPECIES LISTS (for reporting transparency)
  Source: IUCN Red List range maps (mammals + birds)
  CR=Critically Endangered, EN=Endangered, VU=Vulnerable, NT=Near Threatened

  Site: MER_CR_Demarcation2025_final_0000

  Threatened Species (CR/EN/VU) — mammals + birds

    Mammals (13):
      [EN] Nycticebus bengalensis
      [EN] Panthera tigris
      [EN] Elephas maximus
      [EN] Cuon alpinus
      [VU] Bos gaurus
      [VU] Macaca arctoides
      [VU] Macaca leonina
      [VU] Rusa unicolor
      [VU] Catopuma temminckii
      [VU] Neofelis nebulosa
      [VU] Ursus thibetanus
      [VU] Arctictis binturong
      [VU] Panthera pardus

    Birds (31):
      [CR] Perdicula manipurensis
      [CR] Ardea insignis
      [CR] Asarcornis scutulata
      [CR] Emberiza aureola
      [CR] Houbaropsis bengalensis
      [CR] Gyps tenuirostris
      [CR] Rhodonessa caryophyllacea
      [CR] Aythya baeri
 

## 8. Download results

In [ ]:
import os
import json as _json
from google.colab import files

# Augment the report with SoN scoring output
son_output = {
    "metric_concerns": metric_concerns,
    "dim_scores": {
        str(d): {
            "score":       dim_scores.get(d),
            "concern_label": dim_label(dim_scores.get(d)),
            "n_populated": len(dim_populated.get(d, [])),
            "n_total":     len(DIM_MAP[d]),
        }
        for d in [1, 2, 3, 4]
    },
    "son_score":   son_score,
    "son_concern": son_concern_label(son_score),
    "n_dims_used": n_valid,
    "partial_score": partial,
    "threshold_protocol": "Protocol A (published) + Protocol B v1.0 (fixed, Newbold 2016 + GLOBIO4/SBTN basis)",
    "formula": "SoN = (Sigma dim_scores - n_dims) / (n_dims x 4) x 10",
    "is_baseline_run": IS_BASELINE_RUN,
    "project_type": PROJECT_TYPE,
}

combined_path = f"{OUTPUT_DIR}/benchmark_scorecard_with_son.json"
with open(combined_path, "w") as _f:
    _json.dump({**report, "son_scoring": son_output}, _f, indent=2, default=str)

# Baseline + trajectory downloads
if IS_BASELINE_RUN and os.path.exists(BASELINE_FILE):
    files.download(BASELINE_FILE)
if not IS_BASELINE_RUN:
    try:
        traj_path = f"{OUTPUT_DIR}/trajectory.csv"
        df_traj.to_csv(traj_path, index=False)
        files.download(traj_path)
    except NameError:
        pass  # df_traj not defined if Cell 16 was not run

print("Downloading results...")
files.download(f"{OUTPUT_DIR}/benchmark_scorecard.json")
files.download(f"{OUTPUT_DIR}/benchmark_scorecard.csv")
files.download(combined_path)

print("\n\u2713 Downloaded:")
print("  benchmark_scorecard.json          — raw intactness ratios")
print("  benchmark_scorecard.csv           — same, CSV format")
print("  benchmark_scorecard_with_son.json — includes SoN concern levels & score")
if IS_BASELINE_RUN:
    print("  baseline_scorecard.csv             — Year 0 baseline for trajectory tracking")
if not IS_BASELINE_RUN:
    print("  trajectory.csv                     — Year 1+ trajectory labels (Improving/Stable/Declining)")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✓ Downloaded:
  benchmark_scorecard.json          — raw intactness ratios
  benchmark_scorecard.csv           — same, CSV format
  benchmark_scorecard_with_son.json — includes SoN concern levels & score


---

## (Optional) Add a custom indicator

Adding any new indicator requires only a few lines — no pipeline changes needed.

In [ ]:
# Example: Register a custom indicator

def extract_my_indicator(geometry, config):
    """
    Your extraction logic here.
    Return dict with 'value' (float) and 'pixels' (array or None).
    """
    return {"value": 0.42, "pixels": None}

registry.register(
    name="my_indicator",
    display_name="My Custom Indicator",
    source_type="gee",   # or "local_raster", "api", "in_situ"
    extract_fn=extract_my_indicator,
    unit="index",
    value_range=(0.0, 1.0),
    citation="Author et al. (year). DOI:...",
    tier2_eligible=True,   # False for threat/pressure indicators
    higher_is_better=True,
    reference_radius_km=50.0,
    pillar=2,              # TNFD dim: 1=Extent, 2=Condition, 3=Population, 4=Extinction
)

print(f"✓ Registry now has {len(registry)} indicators")
print(f"  Tier 2 eligible: {len(registry.tier2_indicators())}")